# QLoRA Fine-Tuning: Mistral-7B-Instruct

Fine-tunes `mistralai/Mistral-7B-Instruct-v0.3` using 4-bit quantization (QLoRA).

**Runtime required:** GPU → T4 (free tier works)

## Dataset options (set in Cell 3)
| `DATASET_ID` | Purpose |
|---|---|
| `deepset/germanquad` | German Q&A (10k rows) |
| `local_whatsapp` | **Digital avatar** — your own WhatsApp chat data |

For `local_whatsapp`: run `whatsapp_parser.py` locally first, upload the resulting `.jsonl` to Colab, then set the paths in Cell 3.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q -U transformers peft bitsandbytes accelerate datasets trl

In [ ]:
# Cell 2 — Imports + GPU check
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

assert torch.cuda.is_available(), "No GPU found! Change runtime to GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3 — Config

MODEL_ID    = "mistralai/Mistral-7B-Instruct-v0.3"
DATASET_ID  = "local_whatsapp"   # options: deepset/germanquad | local_whatsapp
MAX_ROWS    = 20000

# Only used when DATASET_ID == "local_whatsapp"
# Upload your .jsonl file(s) to Colab, then set paths here
TRAIN_JSONL = "/content/train.jsonl"
VAL_JSONL   = "/content/val.jsonl"

MAX_SEQ_LEN = 512
OUTPUT_DIR  = "./qlora-mistral-output"

In [ ]:
# Cell 4 — Load model in 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — better than fp4 for LLMs
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,     # quantize the quantization constants too
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded successfully")

In [ ]:
# Cell 5 — LoRA adapters
lora_config = LoraConfig(
    r=16,                   # rank — higher = more params, better fit, more VRAM
    lora_alpha=32,          # scaling factor, keep at 2 × r
    target_modules=[        # inject adapters into all linear layers
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.4% trainable params (~27M of 7B total)

In [ ]:
# Cell 6 — Load + format dataset, then 80/10/10 split
from datasets import concatenate_datasets

if DATASET_ID == "deepset/germanquad":
    def format_germanquad(row):
        question = row["question"]
        answer = row["answers"]["text"][0]
        row["text"] = (
            f"<s>[INST] Beantworte die folgende Frage auf Deutsch:\n\n"
            f"{question} [/INST] {answer} </s>"
        )
        return row

    full_dataset = (
        load_dataset("parquet",
                     data_files="hf://datasets/deepset/germanquad@~parquet/plain_text/train/0000.parquet",
                     split="train")
        .map(format_germanquad)
        .select(range(min(MAX_ROWS, 13722)))
    )
    print(f"Sample: {full_dataset[0]['text'][:300]}")

elif DATASET_ID == "local_whatsapp":
    # Combine train + val files, then re-split below
    train_ds = load_dataset("json", data_files=TRAIN_JSONL, split="train")
    val_ds = load_dataset("json", data_files=VAL_JSONL, split="train")
    full_dataset = concatenate_datasets([train_ds, val_ds]).shuffle(seed=42)
    full_dataset = full_dataset.select(range(min(MAX_ROWS, len(full_dataset))))
    print(f"Sample: {full_dataset[0]['text'][:300]}")

# ── 80/10/10 split (train / val / test) ─────────────────────────────────────
split_1 = full_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_1["train"]

split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)
val_dataset = split_2["train"]
test_dataset = split_2["test"]

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# Cell 7 — Train
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=False,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# Cell 8 — Save adapter
trainer.model.save_pretrained(f"{OUTPUT_DIR}/adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")
print(f"Adapter saved to {OUTPUT_DIR}/adapter")

In [ ]:
# Cell 9 — Inference test
model.eval()

prompt = "Hey, was machst du heute Abend?"
inputs = tokenizer(
    f"<s>[INST] {prompt} [/INST]",
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
# Cell 10 — (Optional) Save adapter to Google Drive so it survives session end
from google.colab import drive
drive.mount('/content/drive')
!cp -r {OUTPUT_DIR}/adapter "/content/drive/MyDrive/qlora-mistral-adapter"
print("Saved to Google Drive")